In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from concept_abstraction.training import train_ppo_model, SimpleQEstimator
from concept_abstraction.selection import greedy_selection_supervised
from concept_abstraction.env_utils import *
from concept_abstraction.utils import *
import sys 
import argparse
import secrets
import numpy as np 
import random 
import time 
from collections import Counter
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score


In [3]:
is_jupyter = 'ipykernel' in sys.modules

In [4]:
if is_jupyter: 
    seed        = 42
    num_concepts_selected = 40
    out_folder = "cub"
else:
    parser = argparse.ArgumentParser()
    parser.add_argument('--seed', help='Random Seed', type=int, default=42)
    parser.add_argument('--num_concepts_selected', help='Number of concepts selected by greedy or random',type=int, default=0)
    parser.add_argument('--out_folder', help='Which folder', type=str, default="exploration")

    args = parser.parse_args()

    seed = args.seed
    num_concepts_selected = args.num_concepts_selected
    out_folder = args.out_folder

save_name = secrets.token_hex(4)  

In [5]:
results = {}
results['parameters'] = {'seed'      : seed,
        'num_concepts_selected': num_concepts_selected,
}
print("Parameters {}".format(results['parameters']))

Parameters {'seed': 42, 'num_concepts_selected': 40}


In [6]:
np.random.seed(seed)
random.seed(seed)

In [7]:
dataset = json.load(open("../../data/cub/preprocessed.json"))

## Concept Selection

In [8]:
train_X = np.array([row['attributes'] for row in dataset['train']])
test_X = np.array([row['attributes'] for row in dataset['test']])

In [9]:
all_rows_train = set([''.join([str(int(j)) for j in row]) for row in train_X])
all_rows_test = set([''.join([str(int(j)) for j in row]) for row in test_X])

In [10]:
def get_performance(selected_concepts,accuracy_by_concept):
    train_X = np.array([row['attributes'] for row in dataset['train']])
    test_X = np.array([row['attributes'] for row in dataset['test']])

    flip_probs = 1 - np.array(accuracy_by_concept)
    rand_vals = np.random.rand(*train_X.shape)
    flip_mask = rand_vals < flip_probs  # True means flip
    train_X = np.where(flip_mask, 1 - train_X, train_X)
    train_X = train_X[:,selected_concepts]

    flip_probs = 1 - np.array(accuracy_by_concept)
    rand_vals = np.random.rand(*test_X.shape)
    flip_mask = rand_vals < flip_probs  # True means flip
    test_X = np.where(flip_mask, 1 - test_X, test_X)
    test_X = test_X[:,selected_concepts]


    train_Y = np.array([row['label'] for row in dataset['train']])
    test_Y = np.array([row['label'] for row in dataset['test']])

    mlp = MLPClassifier(
        hidden_layer_sizes=(50),
        activation='relu',
        solver='adam',
        max_iter=1000,  # increase if needed
        random_state=0,
        alpha=1e-3,  # instead of 0.0001,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=20
    )

    # Train the model
    mlp.fit(train_X, train_Y)

    # Predict on the test set
    y_pred = mlp.predict(test_X)

    # Compute accuracy
    acc = accuracy_score(test_Y, y_pred)
    return acc

In [11]:
train_X = np.array([row['attributes'] for row in dataset['train']])
test_X = np.array([row['attributes'] for row in dataset['test']])
train_Y = np.array([row['label'] for row in dataset['train']])
test_Y = np.array([row['label'] for row in dataset['test']])


In [12]:
random_concept_list = [np.random.choice(list(range(train_X.shape[1])),k,replace=False).tolist() for k in range(1,num_concepts_selected)]
random_average_reward = [get_performance(c,np.ones(312)) for c in random_concept_list]

results['random_selection'] = {
    'concepts': random_concept_list, 
    'values': random_average_reward,
}

/usr0/home/naveenr/.local/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


In [35]:
greedy_concept_list = greedy_selection_supervised(train_X,train_Y,num_concepts_selected)
greedy_reward = [get_performance(c,np.ones(312)) for c in greedy_concept_list]
results['greedy_selection'] = {
    'concepts': greedy_concept_list, 
    'values': greedy_reward,
}

In [12]:
manually_selected_concepts = [
    1,
    4,
    6,
    7,
    10,
    14,
    15,
    20,
    21,
    23,
    25,
    29,
    30,
    35,
    36,
    38,
    40,
    44,
    45,
    50,
    51,
    53,
    54,
    56,
    57,
    59,
    63,
    64,
    69,
    70,
    72,
    75,
    80,
    84,
    90,
    91,
    93,
    99,
    101,
    106,
    110,
    111,
    116,
    117,
    119,
    125,
    126,
    131,
    132,
    134,
    145,
    149,
    151,
    152,
    153,
    157,
    158,
    163,
    164,
    168,
    172,
    178,
    179,
    181,
    183,
    187,
    188,
    193,
    194,
    196,
    198,
    202,
    203,
    208,
    209,
    211,
    212,
    213,
    218,
    220,
    221,
    225,
    235,
    236,
    238,
    239,
    240,
    242,
    243,
    244,
    249,
    253,
    254,
    259,
    260,
    262,
    268,
    274,
    277,
    283,
    289,
    292,
    293,
    294,
    298,
    299,
    304,
    305,
    308,
    309,
    310,
    311,
]
results['manually_selected_concepts'] = manually_selected_concepts

In [13]:
random_from_manual_list = [np.random.choice(manually_selected_concepts,k,replace=False).tolist() for k in range(1,num_concepts_selected)]
random_from_manual_reward = [get_performance(c,np.ones(312)) for c in random_from_manual_list]

results['random_manual_selection'] = {
    'concepts': random_from_manual_list, 
    'values': random_from_manual_reward,
}

In [16]:
json.dumps(results['random_manual_selection'])

'{"concepts":[[110],[72,181],[183,23,163],[308,218,172,178],[218,198,260,310,14],[242,110,75,293,309,70],[145,274,260,149,293,277,15],[93,131,193,90,249,244,236,4],[198,213,168,75,38,238,293,15,63],[125,235,163,38,253,196,158,151,208,153],[253,181,172,194,289,7,196,131,292,38,163],[57,178,72,196,36,116,119,211,14,117,59,181],[110,193,7,44,179,64,30,304,15,35,75,209,293],[163,54,90,194,262,45,99,53,91,134,152,188,64,242],[117,101,298,208,125,309,53,116,293,305,209,36,153,70,20],[153,21,208,310,243,188,111,116,194,212,308,54,157,110,132,289],[235,1,56,63,69,157,289,172,178,90,10,93,164,151,54,194,57],[149,274,116,196,30,75,235,194,80,45,283,240,260,193,221,63,178,183],[56,220,212,259,181,111,152,36,253,211,239,238,168,6,15,198,179,309,236],[304,305,253,119,106,110,178,158,221,181,14,164,163,294,153,152,75,1,183,308],[178,116,238,38,1,35,50,196,262,211,64,69,298,244,163,277,40,54,218,209,145],[69,45,30,158,131,152,243,25,163,298,172,274,101,213,106,305,117,72,212,198,157,242],[45,163,289,

In [38]:
results['attribute_names'] = open("../../data/cub/attributes.txt").read().split("\n")

In [39]:
results['random_selection']['values'], results['greedy_selection']['values'], results['random_manual_selection']['values']

([0.009319986192613048],
 [0.005005177770107007, 0.018639972385226095],
 [0.009319986192613048])

In [40]:
# TODO: Actually Train Two-Stage Data

## Save Data

In [47]:
save_path = get_save_path(out_folder,save_name)

In [48]:
delete_duplicate_results(out_folder,"",results)

In [50]:
json.dump(results,open('../../results/'+save_path,'w'))